In [ ]:
import json, os

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project"
PHASE1_DIR = f"{PROJECT_DIR}/phase1_outputs"
PHASE2_DIR = f"{PROJECT_DIR}/phase2_outputs"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_records = load_jsonl(f"{PHASE1_DIR}/train.jsonl")
val_records = load_jsonl(f"{PHASE1_DIR}/val.jsonl")
test_closed_records = load_jsonl(f"{PHASE1_DIR}/test_closed.jsonl")
test_open_records = load_jsonl(f"{PHASE1_DIR}/test_open.jsonl")

with open(f"{PHASE1_DIR}/zero_shot_poets.json", "r", encoding="utf-8") as f:
    zero_shot_poets = json.load(f)

official_closed_set = set(zero_shot_poets["closed_set_poets"])
official_zero_shot = set(zero_shot_poets["zero_shot_poets"])

print("قبل از فیلتر -> train:", len(train_records),
      "| شاعرانِ یکتا:", len(set(r["poet"] for r in train_records)))

train_records = [r for r in train_records if r["poet"] in official_closed_set]
val_records = [r for r in val_records if r["poet"] in official_closed_set]
test_closed_records = [r for r in test_closed_records if r["poet"] in official_closed_set]
test_open_records = [r for r in test_open_records if r["poet"] in official_zero_shot]

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
train_poem_ids = [r["poem_id"] for r in train_records]

val_texts = [r["text"] for r in val_records]
val_poets = [r["poet"] for r in val_records]

test_closed_texts = [r["text"] for r in test_closed_records]
test_closed_poets = [r["poet"] for r in test_closed_records]
test_closed_poem_ids = [r["poem_id"] for r in test_closed_records]
test_closed_ks = [r["k"] for r in test_closed_records]

test_open_texts = [r["text"] for r in test_open_records]
test_open_poets = [r["poet"] for r in test_open_records]
test_open_poem_ids = [r["poem_id"] for r in test_open_records]
test_open_ks = [r["k"] for r in test_open_records]

print("بعد از فیلتر -> train:", len(train_records), "| شاعرانِ یکتا:", len(set(train_poets)), "(باید ۱۵۰ باشد)")
print("val:", len(val_records), "| شاعرانِ یکتا:", len(set(val_poets)))
print("test_closed:", len(test_closed_records), "| شاعرانِ یکتا:", len(set(test_closed_poets)))
print("test_open:", len(test_open_records), "| شاعرانِ یکتا:", len(set(test_open_poets)), "(باید ۳۲ باشد)")

assert set(train_poets) <= official_closed_set
assert set(val_poets) <= official_closed_set
assert set(test_closed_poets) <= official_closed_set
assert set(test_open_poets) <= official_zero_shot
print("✅ اعتبارسنجی موفق: هیچ شاعرِ خارج از لیستِ رسمی وجود ندارد.")

In [ ]:
import numpy as np
from collections import defaultdict

def group_by_k(records):
    by_k = defaultdict(list)
    for r in records:
        by_k[r["k"]].append(r)
    return dict(sorted(by_k.items()))

test_closed_by_k = group_by_k(test_closed_records)
test_open_by_k = group_by_k(test_open_records)

print("=== Test-Closed بر اساسِ k ===")
for k, recs in test_closed_by_k.items():
    print(f"  k={k}: {len(recs):,} نمونه | شاعرانِ یکتا: {len(set(r['poet'] for r in recs))}")

print("\n=== Test-Open بر اساسِ k ===")
for k, recs in test_open_by_k.items():
    print(f"  k={k}: {len(recs):,} نمونه | شاعرانِ یکتا: {len(set(r['poet'] for r in recs))}")

def evaluate_retrieval(query_vecs, query_poets, query_poem_ids,
                        gallery_vecs, gallery_poets, gallery_poem_ids,
                        ks=(1, 5, 10), exclude_same_poem=False, batch_size=512):
    q = np.asarray(query_vecs, dtype=np.float32)
    g = np.asarray(gallery_vecs, dtype=np.float32)
    query_poets = np.asarray(query_poets)
    gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids)
    gallery_poem_ids = np.asarray(gallery_poem_ids)

    q = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-12)
    g = g / (np.linalg.norm(g, axis=1, keepdims=True) + 1e-12)

    max_k = max(ks)
    n_gallery = g.shape[0]
    recall_hits = {k: 0 for k in ks}
    mrr_total = 0.0
    valid_queries = 0

    for start in range(0, q.shape[0], batch_size):
        end = start + batch_size
        batch_q = q[start:end]
        sims = batch_q @ g.T

        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
            sims = np.where(same_poem_mask, -np.inf, sims)

        k_eff = min(max_k, n_gallery - 1) if exclude_same_poem else min(max_k, n_gallery)
        top_idx = np.argpartition(-sims, kth=k_eff - 1, axis=1)[:, :k_eff]
        row_idx = np.arange(top_idx.shape[0])[:, None]
        top_sims = sims[row_idx, top_idx]
        order = np.argsort(-top_sims, axis=1)
        top_idx_sorted = top_idx[row_idx, order]
        top_sims_sorted = top_sims[row_idx, order]

        batch_poets = query_poets[start:end]
        for i in range(end - start if end <= q.shape[0] else q.shape[0] - start):
            if np.all(np.isneginf(top_sims_sorted[i])):
                continue
            valid_queries += 1
            retrieved_poets = gallery_poets[top_idx_sorted[i]]
            hit_positions = np.where(retrieved_poets == batch_poets[i])[0]
            if len(hit_positions) > 0:
                rank = hit_positions[0] + 1
                mrr_total += 1.0 / rank
                for k in ks:
                    if rank <= k:
                        recall_hits[k] += 1

    out = {"n_queries": valid_queries, "mrr": mrr_total / valid_queries if valid_queries else float("nan")}
    for k in ks:
        out[f"recall@{k}"] = recall_hits[k] / valid_queries if valid_queries else float("nan")
    return out

import random
random.seed(0)

sample_gallery = random.sample(train_records, min(3000, len(train_records)))
sample_query = random.sample(test_closed_by_k[1], min(500, len(test_closed_by_k[1])))

print("شاعرانِ یکتا در نمونه‌ی گالری:", len(set(r["poet"] for r in sample_gallery)))
print("شاعرانِ یکتا در نمونه‌ی کوئری:", len(set(r["poet"] for r in sample_query)))

rng = np.random.default_rng(0)
dummy_gallery_vecs = rng.normal(size=(len(sample_gallery), 32))
dummy_query_vecs = rng.normal(size=(len(sample_query), 32))

sanity = evaluate_retrieval(
    query_vecs=dummy_query_vecs,
    query_poets=[r["poet"] for r in sample_query],
    query_poem_ids=[r["poem_id"] for r in sample_query],
    gallery_vecs=dummy_gallery_vecs,
    gallery_poets=[r["poet"] for r in sample_gallery],
    gallery_poem_ids=[r["poem_id"] for r in sample_gallery],
    ks=(1, 5, 10),
    exclude_same_poem=False,
)
print("\n=== خودآزمونِ هارنس (اصلاح‌شده) ===")
print(sanity)

In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import normalize as sk_normalize

def evaluate_retrieval(query_vecs, query_poets, query_poem_ids,
                        gallery_vecs, gallery_poets, gallery_poem_ids,
                        ks=(1, 5, 10), exclude_same_poem=False, batch_size=500):
    def l2_normalize(mat):
        if sp.issparse(mat):
            return sk_normalize(mat, norm="l2", axis=1)
        mat = np.asarray(mat, dtype=np.float32)
        return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)

    q = l2_normalize(query_vecs)
    g = l2_normalize(gallery_vecs)
    query_poets = np.asarray(query_poets)
    gallery_poets = np.asarray(gallery_poets)
    query_poem_ids = np.asarray(query_poem_ids)
    gallery_poem_ids = np.asarray(gallery_poem_ids)

    max_k = max(ks)
    n_gallery = g.shape[0]
    recall_hits = {k: 0 for k in ks}
    mrr_total = 0.0
    valid_queries = 0
    n_queries = q.shape[0]

    for start in range(0, n_queries, batch_size):
        end = min(start + batch_size, n_queries)
        batch_q = q[start:end]
        sims = batch_q @ g.T
        if sp.issparse(sims):
            sims = sims.toarray()
        sims = np.asarray(sims, dtype=np.float32)

        if exclude_same_poem:
            batch_poem_ids = query_poem_ids[start:end]
            same_poem_mask = batch_poem_ids[:, None] == gallery_poem_ids[None, :]
            sims = np.where(same_poem_mask, -np.inf, sims)

        k_eff = min(max_k, n_gallery - 1) if exclude_same_poem else min(max_k, n_gallery)
        top_idx = np.argpartition(-sims, kth=k_eff - 1, axis=1)[:, :k_eff]
        row_idx = np.arange(top_idx.shape[0])[:, None]
        top_sims = sims[row_idx, top_idx]
        order = np.argsort(-top_sims, axis=1)
        top_idx_sorted = top_idx[row_idx, order]
        top_sims_sorted = top_sims[row_idx, order]

        batch_poets = query_poets[start:end]
        for i in range(end - start):
            if np.all(np.isneginf(top_sims_sorted[i])):
                continue
            valid_queries += 1
            retrieved_poets = gallery_poets[top_idx_sorted[i]]
            hit_positions = np.where(retrieved_poets == batch_poets[i])[0]
            if len(hit_positions) > 0:
                rank = hit_positions[0] + 1
                mrr_total += 1.0 / rank
                for k in ks:
                    if rank <= k:
                        recall_hits[k] += 1

    out = {"n_queries": valid_queries, "mrr": mrr_total / valid_queries if valid_queries else float("nan")}
    for k in ks:
        out[f"recall@{k}"] = recall_hits[k] / valid_queries if valid_queries else float("nan")
    return out

In [ ]:
import os, json, time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import LinearSVC

t0 = time.time()

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
train_poem_ids = [r["poem_id"] for r in train_records]

tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), max_features=50000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_texts)
print(f"TF-IDF fit شد. شکلِ ماتریسِ train: {X_train_tfidf.shape}  ({time.time()-t0:.1f}s)")

svc = LinearSVC(C=1.0, max_iter=5000)
svc.fit(X_train_tfidf, train_poets)
X_test_closed_full_tfidf = tfidf.transform([r["text"] for r in test_closed_records])
svc_preds = svc.predict(X_test_closed_full_tfidf)
svc_acc = float(np.mean(np.array(svc_preds) == np.array([r["poet"] for r in test_closed_records])))
print(f"دقتِ طبقه‌بندیِ LinearSVC روی کلِ Test-Closed (فقط Closed-Set، معیارِ کمکی): {svc_acc*100:.2f}%  ({time.time()-t0:.1f}s)")

SVD_DIM = 300
svd = TruncatedSVD(n_components=SVD_DIM, random_state=42)
train_vecs = svd.fit_transform(X_train_tfidf)
print(f"TruncatedSVD fit شد. واریانسِ توضیح‌داده‌شده: {svd.explained_variance_ratio_.sum()*100:.1f}%  ({time.time()-t0:.1f}s)")

def embed_texts(texts):
    return svd.transform(tfidf.transform(texts))

results_table = []
for k in [1, 2, 4, 6]:
    q_closed = test_closed_by_k[k]
    q_closed_vecs = embed_texts([r["text"] for r in q_closed])
    res_closed = evaluate_retrieval(
        query_vecs=q_closed_vecs,
        query_poets=[r["poet"] for r in q_closed],
        query_poem_ids=[r["poem_id"] for r in q_closed],
        gallery_vecs=train_vecs,
        gallery_poets=train_poets,
        gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10),
        exclude_same_poem=False,
    )
    results_table.append({"baseline": "tfidf_linearsvc", "scenario": "closed", "k": k, **res_closed})

    q_open = test_open_by_k[k]
    q_open_vecs = embed_texts([r["text"] for r in q_open])
    res_open = evaluate_retrieval(
        query_vecs=q_open_vecs,
        query_poets=[r["poet"] for r in q_open],
        query_poem_ids=[r["poem_id"] for r in q_open],
        gallery_vecs=q_open_vecs,
        gallery_poets=[r["poet"] for r in q_open],
        gallery_poem_ids=[r["poem_id"] for r in q_open],
        ks=(1, 5, 10),
        exclude_same_poem=True,
    )
    results_table.append({"baseline": "tfidf_linearsvc", "scenario": "open", "k": k, **res_open})

    print(f"k={k} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f} "
          f"| Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}   ({time.time()-t0:.1f}s)")

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs"
BASELINE_DIR = os.path.join(PROJECT_DIR, "tfidf_linearsvc")
os.makedirs(BASELINE_DIR, exist_ok=True)

with open(os.path.join(BASELINE_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump({"svc_closed_accuracy": svc_acc, "retrieval_results": results_table}, f, ensure_ascii=False, indent=2)

all_results_path = os.path.join(PROJECT_DIR, "all_baseline_results.json")
all_results = []
if os.path.exists(all_results_path):
    with open(all_results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
all_results = [r for r in all_results if r.get("baseline") != "tfidf_linearsvc"]
all_results.extend(results_table)
with open(all_results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Baseline 1 کامل شد و ذخیره شد. زمانِ کل: {time.time()-t0:.1f}s")

In [ ]:
import os, json, time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

t0 = time.time()
train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
train_poem_ids = [r["poem_id"] for r in train_records]

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), max_features=50000, sublinear_tf=True)
X_train = vectorizer.fit_transform(train_texts)
print(f"TF-IDF fit شد: {X_train.shape}  ({time.time()-t0:.1f}s)")

clf = LinearSVC(C=1.0, max_iter=5000)
clf.fit(X_train, train_poets)
print(f"LinearSVC آموزش دید  ({time.time()-t0:.1f}s)")

test_closed_texts = [r["text"] for r in test_closed_records]
test_closed_poets = [r["poet"] for r in test_closed_records]
test_closed_poem_ids = [r["poem_id"] for r in test_closed_records]
test_closed_ks = np.array([r["k"] for r in test_closed_records])
X_test_closed = vectorizer.transform(test_closed_texts)
preds = clf.predict(X_test_closed)
acc_by_k = {int(k): accuracy_score(np.array(test_closed_poets)[test_closed_ks == k], preds[test_closed_ks == k])
            for k in sorted(set(test_closed_ks))}
print("دقتِ LinearSVC روی Test-Closed به‌تفکیکِ k:", acc_by_k, f" ({time.time()-t0:.1f}s)")

test_open_texts = [r["text"] for r in test_open_records]
test_open_poets = [r["poet"] for r in test_open_records]
test_open_poem_ids = [r["poem_id"] for r in test_open_records]
test_open_ks = np.array([r["k"] for r in test_open_records])
X_test_open = vectorizer.transform(test_open_texts)

results_table = []
for k_val in [1, 2, 4, 6]:
    idx_c = np.where(test_closed_ks == k_val)[0]
    res_closed = evaluate_retrieval(
        query_vecs=X_test_closed[idx_c],
        query_poets=[test_closed_poets[i] for i in idx_c],
        query_poem_ids=[test_closed_poem_ids[i] for i in idx_c],
        gallery_vecs=X_train, gallery_poets=train_poets, gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10), exclude_same_poem=False, batch_size=500,
    )
    results_table.append({"baseline": "tfidf_linearsvc", "scenario": "closed", "k": k_val, **res_closed})

    idx_o = np.where(test_open_ks == k_val)[0]
    res_open = evaluate_retrieval(
        query_vecs=X_test_open[idx_o],
        query_poets=[test_open_poets[i] for i in idx_o],
        query_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        gallery_vecs=X_test_open[idx_o],
        gallery_poets=[test_open_poets[i] for i in idx_o],
        gallery_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        ks=(1, 5, 10), exclude_same_poem=True, batch_size=500,
    )
    results_table.append({"baseline": "tfidf_linearsvc", "scenario": "open", "k": k_val, **res_open})

    print(f"k={k_val} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f} "
          f"| Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}   ({time.time()-t0:.1f}s)")

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs"
BASELINE_DIR = os.path.join(PROJECT_DIR, "tfidf_linearsvc")
os.makedirs(BASELINE_DIR, exist_ok=True)
with open(os.path.join(BASELINE_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump({"svc_accuracy_by_k": acc_by_k, "retrieval_results": results_table}, f, ensure_ascii=False, indent=2)

all_results_path = os.path.join(PROJECT_DIR, "all_baseline_results.json")
all_results = []
if os.path.exists(all_results_path):
    with open(all_results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
all_results = [r for r in all_results if r.get("baseline") != "tfidf_linearsvc"]
all_results.extend(results_table)
with open(all_results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Baseline 1 (نسخه‌ی درست) کامل شد. زمانِ کل: {time.time()-t0:.1f}s")

In [ ]:
import pprint
pprint.pprint(results_table)

In [ ]:
import time, os, json
import numpy as np
from collections import Counter

t0 = time.time()

def tokenize(text):
    return text.split()

word_counter = Counter()
for text in train_texts:
    word_counter.update(tokenize(text))

MFW_SIZE = 1000
mfw_list = [w for w, _ in word_counter.most_common(MFW_SIZE)]
mfw_index = {w: i for i, w in enumerate(mfw_list)}
print(f"تعدادِ MFW انتخاب‌شده: {len(mfw_list)}  ({time.time()-t0:.1f}s)")

def freq_vector(text):
    tokens = tokenize(text)
    vec = np.zeros(MFW_SIZE, dtype=np.float64)
    if not tokens:
        return vec
    for tok in tokens:
        idx = mfw_index.get(tok)
        if idx is not None:
            vec[idx] += 1
    vec /= len(tokens)
    return vec

def build_matrix(texts):
    mat = np.zeros((len(texts), MFW_SIZE), dtype=np.float64)
    for i, t in enumerate(texts):
        mat[i] = freq_vector(t)
    return mat

X_train_freq = build_matrix(train_texts)
print(f"ماتریسِ فرکانسِ train ساخته شد: {X_train_freq.shape}  ({time.time()-t0:.1f}s)")

mu = X_train_freq.mean(axis=0)
sigma = X_train_freq.std(axis=0)
sigma[sigma == 0] = 1.0

def zscore(mat):
    return (mat - mu) / sigma

train_vecs_delta = zscore(X_train_freq)

def embed_texts_delta(texts):
    return zscore(build_matrix(texts))

results_table_delta = []
for k in [1, 2, 4, 6]:
    q_closed = test_closed_by_k[k]
    q_closed_vecs = embed_texts_delta([r["text"] for r in q_closed])
    res_closed = evaluate_retrieval(
        query_vecs=q_closed_vecs,
        query_poets=[r["poet"] for r in q_closed],
        query_poem_ids=[r["poem_id"] for r in q_closed],
        gallery_vecs=train_vecs_delta,
        gallery_poets=train_poets,
        gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10),
        exclude_same_poem=False,
    )
    results_table_delta.append({"baseline": "burrows_delta", "scenario": "closed", "k": k, **res_closed})

    q_open = test_open_by_k[k]
    q_open_vecs = embed_texts_delta([r["text"] for r in q_open])
    res_open = evaluate_retrieval(
        query_vecs=q_open_vecs,
        query_poets=[r["poet"] for r in q_open],
        query_poem_ids=[r["poem_id"] for r in q_open],
        gallery_vecs=q_open_vecs,
        gallery_poets=[r["poet"] for r in q_open],
        gallery_poem_ids=[r["poem_id"] for r in q_open],
        ks=(1, 5, 10),
        exclude_same_poem=True,
    )
    results_table_delta.append({"baseline": "burrows_delta", "scenario": "open", "k": k, **res_open})

    print(f"k={k} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f} "
          f"| Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}   ({time.time()-t0:.1f}s)")

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs"
BASELINE_DIR = os.path.join(PROJECT_DIR, "burrows_delta")
os.makedirs(BASELINE_DIR, exist_ok=True)

with open(os.path.join(BASELINE_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump({"mfw_size": MFW_SIZE, "retrieval_results": results_table_delta}, f, ensure_ascii=False, indent=2)

all_results_path = os.path.join(PROJECT_DIR, "all_baseline_results.json")
all_results = []
if os.path.exists(all_results_path):
    with open(all_results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
all_results = [r for r in all_results if r.get("baseline") != "burrows_delta"]
all_results.extend(results_table_delta)
with open(all_results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Baseline 2 کامل شد و ذخیره شد. زمانِ کل: {time.time()-t0:.1f}s")

In [ ]:
import pprint
pprint.pprint(results_table_delta)

In [ ]:
import torch
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
val_texts   = [r["text"] for r in val_records]
val_poets   = [r["poet"] for r in val_records]

le = LabelEncoder()
le.fit(train_poets)

assert len(le.classes_) == 150, f"❌ انتظارِ ۱۵۰ کلاس داشتیم، اما {len(le.classes_)} یافت شد!"
assert set(le.classes_) == set(official_closed_set), "❌ برچسب‌هایِ train با official_closed_set یکی نیستند!"
assert set(val_poets) <= set(official_closed_set), "❌ val شاعرِ خارج از official_closed_set دارد!"
print(f"✅ تعدادِ کلاس‌ها: {len(le.classes_)} — دقیقاً منطبق با official_closed_set (۱۵۰ شاعرِ رسمی)")

train_labels = le.transform(train_poets)
val_labels   = le.transform(val_poets)

sample_idx = random.sample(range(len(train_texts)), 2000)
sample_lens = np.array([len(tokenizer.encode(train_texts[i], add_special_tokens=True)) for i in sample_idx])
print("\nتوزیعِ طولِ توکن‌ها (نمونه‌ی ۲۰۰۰تایی از train):")
for p in [50, 75, 90, 95, 99]:
    print(f"  صدکِ {p}: {int(np.percentile(sample_lens, p))} توکن")
print(f"  بیشینه در نمونه: {sample_lens.max()} توکن")

MAX_LENGTH = 256

class PoetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels, self.tokenizer, self.max_length = texts, labels, tokenizer, max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length)
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = PoetDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset   = PoetDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

print(f"\ntrain_dataset: {len(train_dataset)} نمونه | val_dataset: {len(val_dataset)} نمونه")
print("نمونه‌ی رمزگشایی‌شده (۳۰ توکنِ اول):", tokenizer.decode(train_dataset[0]["input_ids"][:30]))

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=150)
print(f"\n✅ مدل بارگذاری شد | num_labels={model.config.num_labels}")

In [ ]:
import inspect
import torch
import transformers
import numpy as np
import time
from transformers import TrainingArguments, Trainer

print("نسخه‌ی transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ندارد")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

MAX_LENGTH = 512
train_dataset.max_length = MAX_LENGTH
val_dataset.max_length = MAX_LENGTH

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"استفاده از bf16: {use_bf16} (در غیرِ این صورت fp16)")

desired_args = dict(
    output_dir="/content/parsbert_baseline3",
    num_train_epochs=8,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=use_bf16,
    fp16=not use_bf16,
    group_by_length=True,
    dataloader_num_workers=8,
    dataloader_persistent_workers=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    torch_compile=True,
)

sig_params = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in sig_params:
    desired_args["eval_strategy"] = "epoch"
elif "evaluation_strategy" in sig_params:
    desired_args["evaluation_strategy"] = "epoch"
if "save_strategy" in sig_params:
    desired_args["save_strategy"] = "epoch"

supported_args = {k: v for k, v in desired_args.items() if k in sig_params}
dropped = set(desired_args) - set(supported_args)
if dropped:
    print(f"⚠️ این آرگومان‌ها در این نسخه پشتیبانی نمی‌شوند و نادیده گرفته شدند: {dropped}")

training_args = TrainingArguments(**supported_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

t0 = time.time()
train_result = trainer.train()
print(f"\n✅ آموزش کامل شد در {time.time()-t0:.1f}s")
print(train_result.metrics)

trainer.save_model("/content/parsbert_baseline3/final")
print("مدلِ نهایی ذخیره شد در /content/parsbert_baseline3/final")

In [ ]:
import shutil
import os
import json

DRIVE_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/parsbert_baseline3"
os.makedirs(DRIVE_DIR, exist_ok=True)

local_model_dir = "/content/parsbert_baseline3/final"
drive_model_dir = os.path.join(DRIVE_DIR, "final")
if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)
shutil.copytree(local_model_dir, drive_model_dir)
print(f"✅ مدل کپی شد به: {drive_model_dir}")

with open(os.path.join(DRIVE_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)
print("✅ متریک‌هایِ آموزش ذخیره شد.")

print("\nمحتویاتِ پوشه‌ی Drive:")
for root, _, files in os.walk(DRIVE_DIR):
    for fn in files:
        print(os.path.join(root, fn))

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import numpy as np
import time
import json
import os
import pprint

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/parsbert_baseline3/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)
embed_model = AutoModel.from_pretrained(DRIVE_MODEL_DIR, output_hidden_states=True).to(device)
embed_model.eval()

EMBED_MAX_LENGTH = 512
EMBED_BATCH_SIZE = 128

@torch.no_grad()
def extract_embeddings(texts, batch_size=EMBED_BATCH_SIZE, max_length=EMBED_MAX_LENGTH):
    all_vecs = []
    use_amp = torch.cuda.is_available()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = embed_tokenizer(
            batch_texts, truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = embed_model(**enc)
        penultimate = out.hidden_states[-2]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        summed = (penultimate.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        mean_pooled = (summed / counts).cpu().numpy()
        all_vecs.append(mean_pooled)
    return np.concatenate(all_vecs, axis=0)

t0 = time.time()
print("در حالِ استخراجِ embeddingِ train (gallery)...")
X_train_emb = extract_embeddings(train_texts)
print(f"train: {X_train_emb.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_closed...")
X_test_closed_emb = extract_embeddings(test_closed_texts)
print(f"test_closed: {X_test_closed_emb.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_open...")
X_test_open_emb = extract_embeddings(test_open_texts)
print(f"test_open: {X_test_open_emb.shape}  ({time.time()-t0:.1f}s)")

parsbert_results = []
for k_val in [1, 2, 4, 6]:
    t0 = time.time()
    idx_c = np.where(test_closed_ks == k_val)[0]
    res_closed = evaluate_retrieval(
        query_vecs=X_test_closed_emb[idx_c],
        query_poets=[test_closed_poets[i] for i in idx_c],
        query_poem_ids=[test_closed_poem_ids[i] for i in idx_c],
        gallery_vecs=X_train_emb,
        gallery_poets=train_poets,
        gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10),
        exclude_same_poem=False,
        batch_size=500,
    )
    res_closed.update({"baseline": "parsbert_classification", "scenario": "closed", "k": k_val})
    parsbert_results.append(res_closed)
    print(f"k={k_val} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f}  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    idx_o = np.where(test_open_ks == k_val)[0]
    res_open = evaluate_retrieval(
        query_vecs=X_test_open_emb[idx_o],
        query_poets=[test_open_poets[i] for i in idx_o],
        query_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        gallery_vecs=X_test_open_emb[idx_o],
        gallery_poets=[test_open_poets[i] for i in idx_o],
        gallery_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        ks=(1, 5, 10),
        exclude_same_poem=True,
        batch_size=500,
    )
    res_open.update({"baseline": "parsbert_classification", "scenario": "open", "k": k_val})
    parsbert_results.append(res_open)
    print(f"k={k_val} | Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}  ({time.time()-t0:.1f}s)")

print("\n=== خلاصه‌ی نتایجِ ParsBERT (Classification, embedding از لایه‌ی ماقبلِ‌آخر) ===")
pprint.pprint(parsbert_results)

PARSBERT_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/parsbert_baseline3"
os.makedirs(PARSBERT_DIR, exist_ok=True)
with open(os.path.join(PARSBERT_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(parsbert_results, f, ensure_ascii=False, indent=2, default=str)
print(f"\n✅ نتایج ذخیره شد در {PARSBERT_DIR}/results.json")

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time, os, json, pprint, shutil
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
backbone = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True).to(device)
backbone.gradient_checkpointing_enable()

class ProjectionHead(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.activation = nn.Tanh()
    def forward(self, x):
        return self.activation(self.dense(x))

proj_head = ProjectionHead(backbone.config.hidden_size).to(device)

train_texts = [r["text"] for r in train_records]
val_texts = [r["text"] for r in val_records]
print(f"تعدادِ متن‌هایِ train: {len(train_texts):,} | val: {len(val_texts):,}")

MAX_LENGTH = 256
BATCH_SIZE = 64
TEMPERATURE = 0.5
NUM_EPOCHS = 3

class SimCSEDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx]

def collate_fn(batch_texts):
    return tokenizer(
        batch_texts, truncation=True, max_length=MAX_LENGTH,
        padding=True, return_tensors="pt",
    )

train_loader = DataLoader(
    SimCSEDataset(train_texts), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=4, drop_last=True,
)
val_loader = DataLoader(
    SimCSEDataset(val_texts), batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=4, drop_last=False,
)
print(f"تعدادِ batchهایِ train در هر epoch: {len(train_loader):,} | val: {len(val_loader):,}")

def mean_pool(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def simcse_loss(z1, z2, temperature=TEMPERATURE):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    sims = z1 @ z2.T / temperature
    labels = torch.arange(sims.size(0), device=sims.device)
    loss_12 = F.cross_entropy(sims, labels)
    loss_21 = F.cross_entropy(sims.T, labels)
    return (loss_12 + loss_21) / 2

optimizer = torch.optim.AdamW(
    list(backbone.parameters()) + list(proj_head.parameters()), lr=3e-5
)
use_amp = torch.cuda.is_available()

print("✅ Setup کامل شد (نسخه‌ی اصلاح‌شده). آماده‌ی سلولِ آموزش.")

In [ ]:

def run_epoch_train(loader, epoch_num):
    backbone.train()
    proj_head.train()
    running_loss, log_every = 0.0, 50
    t0 = time.time()
    for step, enc in enumerate(loader):
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out1 = backbone(**enc)
            out2 = backbone(**enc)
            pooled1 = mean_pool(out1.hidden_states[-2], enc["attention_mask"])
            pooled2 = mean_pool(out2.hidden_states[-2], enc["attention_mask"])
            z1 = proj_head(pooled1.float())
            z2 = proj_head(pooled2.float())
            loss = simcse_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (step + 1) % log_every == 0:
            print(f"epoch={epoch_num} step={step+1}/{len(loader)} "
                  f"train_loss={running_loss/log_every:.4f}  ({time.time()-t0:.1f}s)")
            running_loss = 0.0

@torch.no_grad()
def run_epoch_val(loader):
    backbone.eval()
    proj_head.eval()
    total_loss, n_batches = 0.0, 0
    for enc in loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out1 = backbone(**enc)
            out2 = backbone(**enc)
            pooled1 = mean_pool(out1.hidden_states[-2], enc["attention_mask"])
            pooled2 = mean_pool(out2.hidden_states[-2], enc["attention_mask"])
            z1 = proj_head(pooled1.float())
            z2 = proj_head(pooled2.float())
            loss = simcse_loss(z1, z2)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

DRIVE_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/parsbert_simcse"
os.makedirs(DRIVE_DIR, exist_ok=True)

t0 = time.time()
epoch_val_losses = {}
best_val_loss = float("inf")
best_epoch = None

for epoch in range(1, NUM_EPOCHS + 1):
    run_epoch_train(train_loader, epoch)
    val_loss = run_epoch_val(val_loader)
    epoch_val_losses[epoch] = val_loss
    print(f"✅ epoch={epoch} کامل شد | val_loss={val_loss:.4f}  ({time.time()-t0:.1f}s)")

    epoch_dir_local = f"/content/parsbert_simcse/epoch_{epoch}"
    os.makedirs(epoch_dir_local, exist_ok=True)
    backbone.save_pretrained(epoch_dir_local)
    tokenizer.save_pretrained(epoch_dir_local)

    epoch_dir_drive = os.path.join(DRIVE_DIR, f"epoch_{epoch}")
    if os.path.exists(epoch_dir_drive):
        shutil.rmtree(epoch_dir_drive)
    shutil.copytree(epoch_dir_local, epoch_dir_drive)
    print(f"   checkpoint در Drive ذخیره شد: {epoch_dir_drive}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch

print(f"\n📊 val_loss به‌تفکیکِ epoch: {epoch_val_losses}")
print(f"🏆 بهترین epoch: {best_epoch} (val_loss={best_val_loss:.4f})")

best_epoch_dir_drive = os.path.join(DRIVE_DIR, f"epoch_{best_epoch}")
final_dir_drive = os.path.join(DRIVE_DIR, "final")
if os.path.exists(final_dir_drive):
    shutil.rmtree(final_dir_drive)
shutil.copytree(best_epoch_dir_drive, final_dir_drive)

with open(os.path.join(DRIVE_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(
        {"epoch_val_losses": epoch_val_losses, "best_epoch": best_epoch, "best_val_loss": best_val_loss},
        f, ensure_ascii=False, indent=2,
    )

print(f"\n✅ آموزشِ SimCSE کامل شد. بهترین checkpoint (epoch {best_epoch}) در final/ ذخیره شد. زمانِ کل: {time.time()-t0:.1f}s")

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import numpy as np
import time
import json
import os
import pprint

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/parsbert_simcse/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)
embed_model = AutoModel.from_pretrained(DRIVE_MODEL_DIR, output_hidden_states=True).to(device)
embed_model.eval()

EMBED_MAX_LENGTH = 512
EMBED_BATCH_SIZE = 128

@torch.no_grad()
def extract_embeddings(texts, batch_size=EMBED_BATCH_SIZE, max_length=EMBED_MAX_LENGTH):
    all_vecs = []
    use_amp = torch.cuda.is_available()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = embed_tokenizer(
            batch_texts, truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = embed_model(**enc)
        penultimate = out.hidden_states[-2]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        summed = (penultimate.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        mean_pooled = (summed / counts).cpu().numpy()
        all_vecs.append(mean_pooled)
    return np.concatenate(all_vecs, axis=0)

t0 = time.time()
print("در حالِ استخراجِ embeddingِ train (gallery)...")
X_train_emb_simcse = extract_embeddings(train_texts)
print(f"train: {X_train_emb_simcse.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_closed...")
X_test_closed_emb_simcse = extract_embeddings(test_closed_texts)
print(f"test_closed: {X_test_closed_emb_simcse.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_open...")
X_test_open_emb_simcse = extract_embeddings(test_open_texts)
print(f"test_open: {X_test_open_emb_simcse.shape}  ({time.time()-t0:.1f}s)")

test_closed_ks = np.array([r["k"] for r in test_closed_records])
test_open_ks = np.array([r["k"] for r in test_open_records])

simcse_results = []
for k_val in [1, 2, 4, 6]:
    t0 = time.time()
    idx_c = np.where(test_closed_ks == k_val)[0]
    res_closed = evaluate_retrieval(
        query_vecs=X_test_closed_emb_simcse[idx_c],
        query_poets=[test_closed_poets[i] for i in idx_c],
        query_poem_ids=[test_closed_poem_ids[i] for i in idx_c],
        gallery_vecs=X_train_emb_simcse,
        gallery_poets=train_poets,
        gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10),
        exclude_same_poem=False,
        batch_size=500,
    )
    res_closed.update({"baseline": "parsbert_simcse", "scenario": "closed", "k": k_val})
    simcse_results.append(res_closed)
    print(f"k={k_val} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f}  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    idx_o = np.where(test_open_ks == k_val)[0]
    res_open = evaluate_retrieval(
        query_vecs=X_test_open_emb_simcse[idx_o],
        query_poets=[test_open_poets[i] for i in idx_o],
        query_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        gallery_vecs=X_test_open_emb_simcse[idx_o],
        gallery_poets=[test_open_poets[i] for i in idx_o],
        gallery_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        ks=(1, 5, 10),
        exclude_same_poem=True,
        batch_size=500,
    )
    res_open.update({"baseline": "parsbert_simcse", "scenario": "open", "k": k_val})
    simcse_results.append(res_open)
    print(f"k={k_val} | Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}  ({time.time()-t0:.1f}s)")

print("\n=== خلاصه‌ی نتایجِ ParsBERT+SimCSE (بدونِ‌ناظر، embedding از لایه‌ی ماقبلِ‌آخر) ===")
pprint.pprint(simcse_results)

SIMCSE_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/parsbert_simcse"
os.makedirs(SIMCSE_DIR, exist_ok=True)
with open(os.path.join(SIMCSE_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(simcse_results, f, ensure_ascii=False, indent=2, default=str)
print(f"\n✅ نتایج ذخیره شد در {SIMCSE_DIR}/results.json")

all_results_path = "/content/drive/MyDrive/PACL_Project/phase2_outputs/all_baseline_results.json"
all_results = []
if os.path.exists(all_results_path):
    with open(all_results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
all_results = [r for r in all_results if r.get("baseline") != "parsbert_simcse"]
all_results.extend(simcse_results)
with open(all_results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2, default=str)
print(f"✅ به فایلِ تجمیعی هم اضافه شد: {all_results_path}")

In [ ]:

import torch
import numpy as np
import time, os, json
from transformers import AutoTokenizer, CanineForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

MODEL_NAME = "google/canine-s"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
val_texts   = [r["text"] for r in val_records]
val_poets   = [r["poet"] for r in val_records]

le = LabelEncoder()
le.fit(train_poets)
assert len(le.classes_) == 150, f"❌ انتظارِ ۱۵۰ کلاس داشتیم، اما {len(le.classes_)} یافت شد!"
assert set(le.classes_) == set(official_closed_set), "❌ برچسب‌هایِ train با official_closed_set یکی نیستند!"
assert set(val_poets) <= set(official_closed_set), "❌ val شاعرِ خارج از official_closed_set دارد!"
print(f"✅ تعدادِ کلاس‌ها: {len(le.classes_)} — دقیقاً منطبق با official_closed_set")

train_labels = le.transform(train_poets)
val_labels   = le.transform(val_poets)

import random
sample_idx = random.sample(range(len(train_texts)), 2000)
sample_lens = np.array([len(tokenizer.encode(train_texts[i], add_special_tokens=True)) for i in sample_idx])
print("\nتوزیعِ طولِ توکن‌هایِ CANINE (نمونه‌ی ۲۰۰۰تایی از train):")
for p in [50, 75, 90, 95, 99]:
    print(f"  صدکِ {p}: {int(np.percentile(sample_lens, p))} توکن")
print(f"  بیشینه در نمونه: {sample_lens.max()} توکن")

class PoetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels, self.tokenizer, self.max_length = texts, labels, tokenizer, max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length)
        enc["labels"] = int(self.labels[idx])
        return enc

MAX_LENGTH = 512

train_dataset = PoetDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset   = PoetDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

print(f"\ntrain_dataset: {len(train_dataset)} نمونه | val_dataset: {len(val_dataset)} نمونه")

model = CanineForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=150)
print(f"\n✅ مدلِ CANINE بارگذاری شد | num_labels={model.config.num_labels}")

In [ ]:
import inspect
import torch
import transformers
import numpy as np
import time
from transformers import TrainingArguments, Trainer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"استفاده از bf16: {use_bf16}")

desired_args = dict(
    output_dir="/content/canine_baseline5",
    num_train_epochs=8,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=use_bf16,
    fp16=not use_bf16,
    dataloader_num_workers=8,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
)

sig_params = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in sig_params:
    desired_args["eval_strategy"] = "epoch"
elif "evaluation_strategy" in sig_params:
    desired_args["evaluation_strategy"] = "epoch"
if "save_strategy" in sig_params:
    desired_args["save_strategy"] = "epoch"

supported_args = {k: v for k, v in desired_args.items() if k in sig_params}
training_args = TrainingArguments(**supported_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

t0 = time.time()
train_result = trainer.train()
print(f"\n✅ آموزش کامل شد در {time.time()-t0:.1f}s")
print(train_result.metrics)

trainer.save_model("/content/canine_baseline5/final")
print("مدلِ نهایی ذخیره شد در /content/canine_baseline5/final")

import shutil
DRIVE_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/canine_baseline5"
os.makedirs(DRIVE_DIR, exist_ok=True)
drive_model_dir = os.path.join(DRIVE_DIR, "final")
if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)
shutil.copytree("/content/canine_baseline5/final", drive_model_dir)
with open(os.path.join(DRIVE_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)
print(f"✅ مدل و متریک‌ها در Drive ذخیره شدند: {DRIVE_DIR}")

In [ ]:
from transformers import CanineModel, AutoTokenizer
import torch
import numpy as np
import time, json, os, pprint

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/canine_baseline5/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)
embed_model = CanineModel.from_pretrained(DRIVE_MODEL_DIR, output_hidden_states=True).to(device)
embed_model.eval()

EMBED_MAX_LENGTH = 512
EMBED_BATCH_SIZE = 64

@torch.no_grad()
def extract_embeddings(texts, batch_size=EMBED_BATCH_SIZE, max_length=EMBED_MAX_LENGTH):
    all_vecs = []
    use_amp = torch.cuda.is_available()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = embed_tokenizer(
            batch_texts, truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = embed_model(**enc)
        penultimate = out.hidden_states[-2]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        summed = (penultimate.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        mean_pooled = (summed / counts).cpu().numpy()
        all_vecs.append(mean_pooled)
    return np.concatenate(all_vecs, axis=0)

t0 = time.time()
print("در حالِ استخراجِ embeddingِ train (gallery)...")
X_train_emb_canine = extract_embeddings(train_texts)
print(f"train: {X_train_emb_canine.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_closed...")
X_test_closed_emb_canine = extract_embeddings(test_closed_texts)
print(f"test_closed: {X_test_closed_emb_canine.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_open...")
X_test_open_emb_canine = extract_embeddings(test_open_texts)
print(f"test_open: {X_test_open_emb_canine.shape}  ({time.time()-t0:.1f}s)")

test_closed_ks = np.array([r["k"] for r in test_closed_records])
test_open_ks = np.array([r["k"] for r in test_open_records])

canine_results = []
for k_val in [1, 2, 4, 6]:
    t0 = time.time()
    idx_c = np.where(test_closed_ks == k_val)[0]
    res_closed = evaluate_retrieval(
        query_vecs=X_test_closed_emb_canine[idx_c],
        query_poets=[test_closed_poets[i] for i in idx_c],
        query_poem_ids=[test_closed_poem_ids[i] for i in idx_c],
        gallery_vecs=X_train_emb_canine,
        gallery_poets=train_poets,
        gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10),
        exclude_same_poem=False,
        batch_size=500,
    )
    res_closed.update({"baseline": "canine", "scenario": "closed", "k": k_val})
    canine_results.append(res_closed)
    print(f"k={k_val} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f}  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    idx_o = np.where(test_open_ks == k_val)[0]
    res_open = evaluate_retrieval(
        query_vecs=X_test_open_emb_canine[idx_o],
        query_poets=[test_open_poets[i] for i in idx_o],
        query_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        gallery_vecs=X_test_open_emb_canine[idx_o],
        gallery_poets=[test_open_poets[i] for i in idx_o],
        gallery_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        ks=(1, 5, 10),
        exclude_same_poem=True,
        batch_size=500,
    )
    res_open.update({"baseline": "canine", "scenario": "open", "k": k_val})
    canine_results.append(res_open)
    print(f"k={k_val} | Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}  ({time.time()-t0:.1f}s)")

print("\n=== خلاصه‌ی نتایجِ CANINE ===")
pprint.pprint(canine_results)

CANINE_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/canine_baseline5"
with open(os.path.join(CANINE_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(canine_results, f, ensure_ascii=False, indent=2, default=str)
print(f"\n✅ نتایج ذخیره شد در {CANINE_DIR}/results.json")

all_results_path = "/content/drive/MyDrive/PACL_Project/phase2_outputs/all_baseline_results.json"
all_results = []
if os.path.exists(all_results_path):
    with open(all_results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
all_results = [r for r in all_results if r.get("baseline") != "canine"]
all_results.extend(canine_results)
with open(all_results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2, default=str)
print(f"✅ به فایلِ تجمیعی هم اضافه شد: {all_results_path}")

In [ ]:

import torch
import torch.nn as nn
import random
import numpy as np
from transformers import AutoTokenizer, T5EncoderModel, DataCollatorWithPadding
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

MODEL_NAME = "google/byt5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
val_texts   = [r["text"] for r in val_records]
val_poets   = [r["poet"] for r in val_records]

le = LabelEncoder()
le.fit(train_poets)
assert len(le.classes_) == 150, f"❌ انتظارِ ۱۵۰ کلاس داشتیم، اما {len(le.classes_)} یافت شد!"
assert set(le.classes_) == set(official_closed_set), "❌ برچسب‌هایِ train با official_closed_set یکی نیستند!"
assert set(val_poets) <= set(official_closed_set), "❌ val شاعرِ خارج از official_closed_set دارد!"
print(f"✅ تعدادِ کلاس‌ها: {len(le.classes_)} — دقیقاً منطبق با official_closed_set")

train_labels = le.transform(train_poets)
val_labels   = le.transform(val_poets)

sample_idx = random.sample(range(len(train_texts)), 2000)
sample_lens = np.array([len(tokenizer.encode(train_texts[i], add_special_tokens=True)) for i in sample_idx])
print("\nتوزیعِ طولِ توکن‌هایِ ByT5 (نمونه‌ی ۲۰۰۰تایی از train):")
for p in [50, 75, 90, 95, 99]:
    print(f"  صدکِ {p}: {int(np.percentile(sample_lens, p))} توکن")
print(f"  بیشینه در نمونه: {sample_lens.max()} توکن")

class PoetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels, self.tokenizer, self.max_length = texts, labels, tokenizer, max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length)
        enc["labels"] = int(self.labels[idx])
        return enc

MAX_LENGTH = 512

train_dataset = PoetDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset   = PoetDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

print(f"\ntrain_dataset: {len(train_dataset)} نمونه | val_dataset: {len(val_dataset)} نمونه")

class ByT5ForClassification(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.d_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        last_hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        pooled = summed / counts
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {"loss": loss, "logits": logits, "hidden_states": out.hidden_states}

model = ByT5ForClassification(MODEL_NAME, num_labels=150)
print(f"\n✅ مدلِ ByT5-Base (Encoder + Classification Head) بارگذاری شد | num_labels=150")

In [ ]:
import torch
import numpy as np
import time, os, json, shutil
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

BATCH_SIZE = 32
NUM_EPOCHS = 8
LR = 2e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=data_collator)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
use_amp = torch.cuda.is_available()

def run_epoch_train(loader, epoch_num):
    model.train()
    running_loss, log_every = 0.0, 100
    t0 = time.time()
    for step, batch in enumerate(loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"])
            loss = out["loss"]
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (step + 1) % log_every == 0:
            print(f"epoch={epoch_num} step={step+1}/{len(loader)} train_loss={running_loss/log_every:.4f}  ({time.time()-t0:.1f}s)")
            running_loss = 0.0

@torch.no_grad()
def run_epoch_val(loader):
    model.eval()
    total_loss, n_batches, correct, total = 0.0, 0, 0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["labels"])
        total_loss += out["loss"].item()
        n_batches += 1
        preds = out["logits"].argmax(dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)
    return total_loss / n_batches, correct / total

DRIVE_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/byt5_baseline6"
os.makedirs(DRIVE_DIR, exist_ok=True)

t0 = time.time()
epoch_metrics = {}
best_val_acc = -1
best_epoch = None

for epoch in range(1, NUM_EPOCHS + 1):
    run_epoch_train(train_loader, epoch)
    val_loss, val_acc = run_epoch_val(val_loader)
    epoch_metrics[epoch] = {"val_loss": val_loss, "val_accuracy": val_acc}
    print(f"✅ epoch={epoch} کامل شد | val_loss={val_loss:.4f} | val_accuracy={val_acc*100:.2f}%  ({time.time()-t0:.1f}s)")

    epoch_dir_local = f"/content/byt5_baseline6/epoch_{epoch}"
    os.makedirs(epoch_dir_local, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(epoch_dir_local, "model.pt"))
    tokenizer.save_pretrained(epoch_dir_local)

    epoch_dir_drive = os.path.join(DRIVE_DIR, f"epoch_{epoch}")
    if os.path.exists(epoch_dir_drive):
        shutil.rmtree(epoch_dir_drive)
    shutil.copytree(epoch_dir_local, epoch_dir_drive)
    print(f"   checkpoint در Drive ذخیره شد: {epoch_dir_drive}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

print(f"\n📊 متریک‌ها به‌تفکیکِ epoch: {epoch_metrics}")
print(f"🏆 بهترین epoch: {best_epoch} (val_accuracy={best_val_acc*100:.2f}%)")

best_epoch_dir_drive = os.path.join(DRIVE_DIR, f"epoch_{best_epoch}")
final_dir_drive = os.path.join(DRIVE_DIR, "final")
if os.path.exists(final_dir_drive):
    shutil.rmtree(final_dir_drive)
shutil.copytree(best_epoch_dir_drive, final_dir_drive)

with open(os.path.join(DRIVE_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"epoch_metrics": epoch_metrics, "best_epoch": best_epoch, "best_val_accuracy": best_val_acc}, f, ensure_ascii=False, indent=2)

print(f"\n✅ آموزشِ ByT5-Base کامل شد. بهترین checkpoint (epoch {best_epoch}) در final/ ذخیره شد. زمانِ کل: {time.time()-t0:.1f}s")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time, json, os, pprint
from transformers import AutoTokenizer, T5EncoderModel

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/byt5_baseline6/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)

class ByT5ForClassification(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.d_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        return out

embed_model = ByT5ForClassification("google/byt5-base", num_labels=150)
state_dict = torch.load(os.path.join(DRIVE_MODEL_DIR, "model.pt"), map_location=device)
embed_model.load_state_dict(state_dict)
embed_model = embed_model.to(device)
embed_model.eval()

EMBED_MAX_LENGTH = 512
EMBED_BATCH_SIZE = 32

@torch.no_grad()
def extract_embeddings(texts, batch_size=EMBED_BATCH_SIZE, max_length=EMBED_MAX_LENGTH):
    all_vecs = []
    use_amp = torch.cuda.is_available()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = embed_tokenizer(
            batch_texts, truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            out = embed_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
        penultimate = out.hidden_states[-2]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        summed = (penultimate.float() * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        mean_pooled = (summed / counts).cpu().numpy()
        all_vecs.append(mean_pooled)
    return np.concatenate(all_vecs, axis=0)

t0 = time.time()
print("در حالِ استخراجِ embeddingِ train (gallery)...")
X_train_emb_byt5 = extract_embeddings(train_texts)
print(f"train: {X_train_emb_byt5.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_closed...")
X_test_closed_emb_byt5 = extract_embeddings(test_closed_texts)
print(f"test_closed: {X_test_closed_emb_byt5.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_open...")
X_test_open_emb_byt5 = extract_embeddings(test_open_texts)
print(f"test_open: {X_test_open_emb_byt5.shape}  ({time.time()-t0:.1f}s)")

test_closed_ks = np.array([r["k"] for r in test_closed_records])
test_open_ks = np.array([r["k"] for r in test_open_records])

byt5_results = []
for k_val in [1, 2, 4, 6]:
    t0 = time.time()
    idx_c = np.where(test_closed_ks == k_val)[0]
    res_closed = evaluate_retrieval(
        query_vecs=X_test_closed_emb_byt5[idx_c],
        query_poets=[test_closed_poets[i] for i in idx_c],
        query_poem_ids=[test_closed_poem_ids[i] for i in idx_c],
        gallery_vecs=X_train_emb_byt5,
        gallery_poets=train_poets,
        gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10),
        exclude_same_poem=False,
        batch_size=500,
    )
    res_closed.update({"baseline": "byt5_base", "scenario": "closed", "k": k_val})
    byt5_results.append(res_closed)
    print(f"k={k_val} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f}  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    idx_o = np.where(test_open_ks == k_val)[0]
    res_open = evaluate_retrieval(
        query_vecs=X_test_open_emb_byt5[idx_o],
        query_poets=[test_open_poets[i] for i in idx_o],
        query_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        gallery_vecs=X_test_open_emb_byt5[idx_o],
        gallery_poets=[test_open_poets[i] for i in idx_o],
        gallery_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        ks=(1, 5, 10),
        exclude_same_poem=True,
        batch_size=500,
    )
    res_open.update({"baseline": "byt5_base", "scenario": "open", "k": k_val})
    byt5_results.append(res_open)
    print(f"k={k_val} | Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}  ({time.time()-t0:.1f}s)")

print("\n=== خلاصه‌ی نتایجِ ByT5-Base ===")
pprint.pprint(byt5_results)

BYT5_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/byt5_baseline6"
with open(os.path.join(BYT5_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(byt5_results, f, ensure_ascii=False, indent=2, default=str)
print(f"\n✅ نتایج ذخیره شد در {BYT5_DIR}/results.json")

all_results_path = "/content/drive/MyDrive/PACL_Project/phase2_outputs/all_baseline_results.json"
all_results = []
if os.path.exists(all_results_path):
    with open(all_results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
all_results = [r for r in all_results if r.get("baseline") != "byt5_base"]
all_results.extend(byt5_results)
with open(all_results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2, default=str)
print(f"✅ به فایلِ تجمیعی هم اضافه شد: {all_results_path}")

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from collections import defaultdict
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = [r["text"] for r in train_records]
train_poets = [r["poet"] for r in train_records]
val_texts   = [r["text"] for r in val_records]
val_poets   = [r["poet"] for r in val_records]

assert set(train_poets) == set(official_closed_set), "❌ برچسب‌هایِ train با official_closed_set یکی نیستند!"
assert set(val_poets) <= set(official_closed_set), "❌ val شاعرِ خارج از official_closed_set دارد!"
print(f"✅ تعدادِ شاعرانِ یکتا در train: {len(set(train_poets))} — دقیقاً منطبق با official_closed_set")

poet_to_indices = defaultdict(list)
for idx, poet in enumerate(train_poets):
    poet_to_indices[poet].append(idx)

eligible_poets = [p for p, idxs in poet_to_indices.items() if len(idxs) >= 2]
print(f"شاعرانِ واجدِ شرایط برایِ contrastive (>=۲ نمونه): {len(eligible_poets)} از {len(poet_to_indices)}")

class AttentivePooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, attention_mask):
        scores = self.attention(hidden_states).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        pooled = (hidden_states * weights).sum(dim=1)
        return pooled

class LUARModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        hidden_size = self.encoder.config.hidden_size
        self.attentive_pool = AttentivePooling(hidden_size)
        self.projection = nn.Linear(hidden_size, hidden_size)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.attentive_pool(out.hidden_states[-2], attention_mask)
        z = self.projection(pooled)
        return z

model = LUARModel(MODEL_NAME).to(device)
print(f"\n✅ مدلِ LUAR (ParsBERT + Attentive Pooling) بارگذاری شد")

MAX_LENGTH = 512

In [ ]:

import torch
import torch.nn.functional as F
import numpy as np
import random
import time, os, json, shutil
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, Sampler

POETS_PER_BATCH = 8
SAMPLES_PER_POET = 4
BATCH_SIZE = POETS_PER_BATCH * SAMPLES_PER_POET
TEMPERATURE = 0.1
NUM_EPOCHS = 8

class PoemDataset(Dataset):
    def __init__(self, texts, poets):
        self.texts, self.poets = texts, poets
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.poets[idx]

class EpisodicBatchSampler(Sampler):
    def __init__(self, poet_to_indices, eligible_poets, poets_per_batch, samples_per_poet, num_batches):
        self.poet_to_indices = poet_to_indices
        self.eligible_poets = eligible_poets
        self.poets_per_batch = poets_per_batch
        self.samples_per_poet = samples_per_poet
        self.num_batches = num_batches

    def __iter__(self):
        for _ in range(self.num_batches):
            chosen_poets = random.sample(self.eligible_poets, self.poets_per_batch)
            batch_indices = []
            for poet in chosen_poets:
                idxs = self.poet_to_indices[poet]
                if len(idxs) >= self.samples_per_poet:
                    chosen = random.sample(idxs, self.samples_per_poet)
                else:
                    chosen = random.choices(idxs, k=self.samples_per_poet)
                batch_indices.extend(chosen)
            yield batch_indices

    def __len__(self):
        return self.num_batches

def collate_fn(batch):
    texts = [t for t, p in batch]
    poets = [p for t, p in batch]
    enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")
    return enc, poets

train_dataset = PoemDataset(train_texts, train_poets)
NUM_BATCHES_PER_EPOCH = len(train_texts) // BATCH_SIZE
sampler = EpisodicBatchSampler(poet_to_indices, eligible_poets, POETS_PER_BATCH, SAMPLES_PER_POET, NUM_BATCHES_PER_EPOCH)
train_loader = DataLoader(train_dataset, batch_sampler=sampler, collate_fn=collate_fn, num_workers=4)

val_poet_to_indices = defaultdict(list)
for idx, poet in enumerate(val_poets):
    val_poet_to_indices[poet].append(idx)
val_eligible_poets = [p for p, idxs in val_poet_to_indices.items() if len(idxs) >= 2]
val_dataset = PoemDataset(val_texts, val_poets)
NUM_VAL_BATCHES = max(1, len(val_texts) // BATCH_SIZE)
val_sampler = EpisodicBatchSampler(val_poet_to_indices, val_eligible_poets, POETS_PER_BATCH, SAMPLES_PER_POET, NUM_VAL_BATCHES)
val_loader = DataLoader(val_dataset, batch_sampler=val_sampler, collate_fn=collate_fn, num_workers=4)

print(f"تعدادِ batch در هر epoch (train): {len(train_loader)} | (val): {len(val_loader)}")

def supervised_contrastive_loss(embeddings, poets, temperature=TEMPERATURE):
    z = F.normalize(embeddings, dim=-1)
    sims = z @ z.T / temperature
    batch_size = z.size(0)

    poets_arr = np.array(poets)
    same_poet_mask = torch.tensor(
        poets_arr[:, None] == poets_arr[None, :], device=z.device, dtype=torch.bool
    )
    self_mask = torch.eye(batch_size, device=z.device, dtype=torch.bool)
    positive_mask = same_poet_mask & ~self_mask

    NEG_INF = -1e9
    sims = sims.masked_fill(self_mask, NEG_INF)

    log_prob = sims - torch.logsumexp(sims, dim=1, keepdim=True)

    pos_counts = positive_mask.sum(dim=1).clamp(min=1)
    loss_per_sample = -(log_prob * positive_mask.float()).sum(dim=1) / pos_counts
    valid = positive_mask.sum(dim=1) > 0

    if valid.sum() == 0:
        return torch.tensor(0.0, device=z.device, requires_grad=True)

    return loss_per_sample[valid].mean()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
use_amp = torch.cuda.is_available()

def run_epoch_train(loader, epoch_num):
    model.train()
    running_loss, log_every = 0.0, 50
    t0 = time.time()
    for step, (enc, poets) in enumerate(loader):
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            z = model(enc["input_ids"], enc["attention_mask"])
        loss = supervised_contrastive_loss(z.float(), poets)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        if (step + 1) % log_every == 0:
            print(f"epoch={epoch_num} step={step+1}/{len(loader)} train_loss={running_loss/log_every:.4f}  ({time.time()-t0:.1f}s)")
            running_loss = 0.0

@torch.no_grad()
def run_epoch_val(loader):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for enc, poets in loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            z = model(enc["input_ids"], enc["attention_mask"])
        loss = supervised_contrastive_loss(z.float(), poets)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

DRIVE_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/luar_baseline7"
os.makedirs(DRIVE_DIR, exist_ok=True)

t0 = time.time()
epoch_val_losses = {}
best_val_loss = float("inf")
best_epoch = None

for epoch in range(1, NUM_EPOCHS + 1):
    run_epoch_train(train_loader, epoch)
    val_loss = run_epoch_val(val_loader)
    epoch_val_losses[epoch] = val_loss
    print(f"✅ epoch={epoch} کامل شد | val_loss={val_loss:.4f}  ({time.time()-t0:.1f}s)")

    epoch_dir_local = f"/content/luar_baseline7/epoch_{epoch}"
    os.makedirs(epoch_dir_local, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(epoch_dir_local, "model.pt"))
    tokenizer.save_pretrained(epoch_dir_local)

    epoch_dir_drive = os.path.join(DRIVE_DIR, f"epoch_{epoch}")
    if os.path.exists(epoch_dir_drive):
        shutil.rmtree(epoch_dir_drive)
    shutil.copytree(epoch_dir_local, epoch_dir_drive)
    print(f"   checkpoint در Drive ذخیره شد: {epoch_dir_drive}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch

print(f"\n📊 val_loss به‌تفکیکِ epoch: {epoch_val_losses}")
print(f"🏆 بهترین epoch: {best_epoch} (val_loss={best_val_loss:.4f})")

best_epoch_dir_drive = os.path.join(DRIVE_DIR, f"epoch_{best_epoch}")
final_dir_drive = os.path.join(DRIVE_DIR, "final")
if os.path.exists(final_dir_drive):
    shutil.rmtree(final_dir_drive)
shutil.copytree(best_epoch_dir_drive, final_dir_drive)

with open(os.path.join(DRIVE_DIR, "train_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"epoch_val_losses": epoch_val_losses, "best_epoch": best_epoch, "best_val_loss": best_val_loss}, f, ensure_ascii=False, indent=2)

print(f"\n✅ آموزشِ LUAR کامل شد. بهترین checkpoint (epoch {best_epoch}) در final/ ذخیره شد. زمانِ کل: {time.time()-t0:.1f}s")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time, json, os, pprint
from transformers import AutoTokenizer, AutoModel

DRIVE_MODEL_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/luar_baseline7/final"
device = "cuda" if torch.cuda.is_available() else "cpu"

embed_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)

class AttentivePooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, attention_mask):
        scores = self.attention(hidden_states).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        pooled = (hidden_states * weights).sum(dim=1)
        return pooled

class LUARModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        hidden_size = self.encoder.config.hidden_size
        self.attentive_pool = AttentivePooling(hidden_size)
        self.projection = nn.Linear(hidden_size, hidden_size)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.attentive_pool(out.hidden_states[-2], attention_mask)
        z = self.projection(pooled)
        return z

embed_model = LUARModel("HooshvareLab/bert-fa-base-uncased")
state_dict = torch.load(os.path.join(DRIVE_MODEL_DIR, "model.pt"), map_location=device)
embed_model.load_state_dict(state_dict)
embed_model = embed_model.to(device)
embed_model.eval()

EMBED_MAX_LENGTH = 512
EMBED_BATCH_SIZE = 128

@torch.no_grad()
def extract_embeddings(texts, batch_size=EMBED_BATCH_SIZE, max_length=EMBED_MAX_LENGTH):
    all_vecs = []
    use_amp = torch.cuda.is_available()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = embed_tokenizer(
            batch_texts, truncation=True, max_length=max_length,
            padding=True, return_tensors="pt",
        ).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            z = embed_model(enc["input_ids"], enc["attention_mask"])
        all_vecs.append(z.float().cpu().numpy())
    return np.concatenate(all_vecs, axis=0)

t0 = time.time()
print("در حالِ استخراجِ embeddingِ train (gallery)...")
X_train_emb_luar = extract_embeddings(train_texts)
print(f"train: {X_train_emb_luar.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_closed...")
X_test_closed_emb_luar = extract_embeddings(test_closed_texts)
print(f"test_closed: {X_test_closed_emb_luar.shape}  ({time.time()-t0:.1f}s)")

t0 = time.time()
print("در حالِ استخراجِ embeddingِ test_open...")
X_test_open_emb_luar = extract_embeddings(test_open_texts)
print(f"test_open: {X_test_open_emb_luar.shape}  ({time.time()-t0:.1f}s)")

test_closed_ks = np.array([r["k"] for r in test_closed_records])
test_open_ks = np.array([r["k"] for r in test_open_records])

luar_results = []
for k_val in [1, 2, 4, 6]:
    t0 = time.time()
    idx_c = np.where(test_closed_ks == k_val)[0]
    res_closed = evaluate_retrieval(
        query_vecs=X_test_closed_emb_luar[idx_c],
        query_poets=[test_closed_poets[i] for i in idx_c],
        query_poem_ids=[test_closed_poem_ids[i] for i in idx_c],
        gallery_vecs=X_train_emb_luar,
        gallery_poets=train_poets,
        gallery_poem_ids=train_poem_ids,
        ks=(1, 5, 10),
        exclude_same_poem=False,
        batch_size=500,
    )
    res_closed.update({"baseline": "luar", "scenario": "closed", "k": k_val})
    luar_results.append(res_closed)
    print(f"k={k_val} | Closed: R@1={res_closed['recall@1']*100:.2f}% MRR={res_closed['mrr']:.3f}  ({time.time()-t0:.1f}s)")

    t0 = time.time()
    idx_o = np.where(test_open_ks == k_val)[0]
    res_open = evaluate_retrieval(
        query_vecs=X_test_open_emb_luar[idx_o],
        query_poets=[test_open_poets[i] for i in idx_o],
        query_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        gallery_vecs=X_test_open_emb_luar[idx_o],
        gallery_poets=[test_open_poets[i] for i in idx_o],
        gallery_poem_ids=[test_open_poem_ids[i] for i in idx_o],
        ks=(1, 5, 10),
        exclude_same_poem=True,
        batch_size=500,
    )
    res_open.update({"baseline": "luar", "scenario": "open", "k": k_val})
    luar_results.append(res_open)
    print(f"k={k_val} | Open: R@1={res_open['recall@1']*100:.2f}% MRR={res_open['mrr']:.3f}  ({time.time()-t0:.1f}s)")

print("\n=== خلاصه‌ی نتایجِ LUAR ===")
pprint.pprint(luar_results)

LUAR_DIR = "/content/drive/MyDrive/PACL_Project/phase2_outputs/luar_baseline7"
with open(os.path.join(LUAR_DIR, "results.json"), "w", encoding="utf-8") as f:
    json.dump(luar_results, f, ensure_ascii=False, indent=2, default=str)
print(f"\n✅ نتایج ذخیره شد در {LUAR_DIR}/results.json")

all_results_path = "/content/drive/MyDrive/PACL_Project/phase2_outputs/all_baseline_results.json"
all_results = []
if os.path.exists(all_results_path):
    with open(all_results_path, "r", encoding="utf-8") as f:
        all_results = json.load(f)
all_results = [r for r in all_results if r.get("baseline") != "luar"]
all_results.extend(luar_results)
with open(all_results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2, default=str)
print(f"✅ به فایلِ تجمیعی هم اضافه شد: {all_results_path}")

In [ ]:
import json, os

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project"
PHASE1_DIR = os.path.join(PROJECT_DIR, "phase1_outputs")

with open(os.path.join(PHASE1_DIR, "zero_shot_poets.json"), "r", encoding="utf-8") as f:
    zs_data = json.load(f)

closed_set_poets = zs_data["closed_set_poets"]
open_set_poets = zs_data["zero_shot_poets"]

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_closed = load_jsonl(os.path.join(PHASE1_DIR, "test_closed.jsonl"))
test_open = load_jsonl(os.path.join(PHASE1_DIR, "test_open.jsonl"))
print(f"Closed candidates: {len(closed_set_poets)} | Open candidates: {len(open_set_poets)}")
print(f"test_closed: {len(test_closed)} | test_open: {len(test_open)}")

In [ ]:
import random
from collections import defaultdict, Counter

random.seed(42)
K_ALLOCATION = {1: 400, 2: 400, 4: 100, 6: 100}

def stratified_sample_by_poet(rows, k, n_target):
    pool = [r for r in rows if r["k"] == k]
    by_poet = defaultdict(list)
    for r in pool:
        by_poet[r["poet"]].append(r)
    poets = list(by_poet.keys())
    random.shuffle(poets)
    sampled, idx_per_poet = [], {p: 0 for p in poets}
    while len(sampled) < n_target and poets:
        progressed = False
        for p in list(poets):
            lst = by_poet[p]
            i = idx_per_poet[p]
            if i < len(lst):
                sampled.append(lst[i]); idx_per_poet[p] += 1; progressed = True
                if len(sampled) >= n_target:
                    break
        if not progressed:
            break
    return sampled

llm_sample = []
for k, total_n in K_ALLOCATION.items():
    n_each = total_n // 2
    llm_sample.extend(stratified_sample_by_poet(test_closed, k, n_each))
    llm_sample.extend(stratified_sample_by_poet(test_open, k, n_each))

print(f"Total sampled: {len(llm_sample)}")
print(Counter((r["split"], r["k"]) for r in llm_sample))

LLM_DIR = os.path.join(PROJECT_DIR, "phase2_outputs", "llm_baseline8")
os.makedirs(LLM_DIR, exist_ok=True)
with open(os.path.join(LLM_DIR, "llm_baseline8_sample.jsonl"), "w", encoding="utf-8") as f:
    for r in llm_sample:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
def build_candidate_list(row):
    return closed_set_poets if row["split"] == "test_closed" else open_set_poets

def format_candidates(candidates):
    return "\n".join(f"{i+1}. {name}" for i, name in enumerate(candidates))

def build_messages(row):
    candidates = build_candidate_list(row)
    candidate_str = format_candidates(candidates)
    system_msg = (
        "You are an expert in Persian (Farsi) classical poetry stylometry. "
        "Given a short excerpt from a Persian poem and a numbered list of candidate poets, "
        "identify which poet most likely wrote it, based on subtle stylistic cues: meter (وزن), "
        "rhyme/refrain (قافیه/ردیف), diction and vocabulary, era-specific language, and thematic tone.\n\n"
        "Respond with STRICTLY VALID JSON ONLY. No markdown, no code fences, no text before or after "
        "the JSON object. Use exactly this schema:\n"
        '{\n  "ranked_authors": ["<name>", "... exactly 10 names total"],\n  "rationale": "1-3 sentences"\n}\n\n'
        "Rules:\n"
        "- \"ranked_authors\" must contain EXACTLY 10 distinct names, ordered from most to least likely.\n"
        "- Every name MUST be copied character-for-character from the numbered candidate list below.\n"
        "- Do not invent names that are not in the candidate list.\n"
        "- Do not repeat any name."
    )
    user_msg = (
        f"Persian poem excerpt ({row['k']} couplet(s)):\n\"\"\"\n{row['text']}\n\"\"\"\n\n"
        f"Candidate list ({len(candidates)} poets):\n{candidate_str}\n\n"
        "Task: Rank the 10 most likely authors from the candidate list above (most likely first), "
        "and briefly explain the stylistic rationale for your top choice."
    )
    return [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]

In [ ]:
from openai import OpenAI
from getpass import getpass

api_keys = []
for i in range(3):
    key = getpass(f"API key شماره‌ی {i+1} را وارد کنید (مخفی می‌ماند): ")
    api_keys.append(key)

clients = [OpenAI(base_url="https://api.tokenmix.ai/v1", api_key=k) for k in api_keys]
print(f"{len(clients)} کلاینت ساخته شد.")

In [ ]:
import re

def normalize_name(name):
    return name.replace("\u200c", "").replace(" ", "").strip()

def extract_json(raw_text):
    if raw_text is None:
        return None
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                return None
        return None

def is_structurally_valid(parsed):
    if not isinstance(parsed, dict):
        return False
    authors = parsed.get("ranked_authors")
    if not isinstance(authors, list) or len(authors) != 10:
        return False
    if not all(isinstance(a, str) and a.strip() for a in authors):
        return False
    if len(set(normalize_name(a) for a in authors)) != 10:
        return False
    if not parsed.get("rationale"):
        return False
    return True

In [ ]:
import os, time

LLM_DIR = os.path.join(PROJECT_DIR, "phase2_outputs", "llm_baseline8")
os.makedirs(LLM_DIR, exist_ok=True)
raw_output_path = os.path.join(LLM_DIR, "llama_zeroshot_raw.jsonl")

if os.path.exists(raw_output_path):
    os.remove(raw_output_path)
    print("فایلِ قبلی پاک شد؛ شروعِ اجرا از صفر.")

def make_sample_key(row):
    return f"{row['poem_id']}_{row['k']}_{row['start_unit_idx']}"

already_done = set()
MAX_RETRIES = 3

with open(raw_output_path, "a", encoding="utf-8") as out_f:
    for i, row in enumerate(llm_sample):
        key = make_sample_key(row)
        if key in already_done:
            continue

        messages = build_messages(row)
        raw, valid = None, False
        for attempt in range(MAX_RETRIES):
            client = clients[(i + attempt) % len(clients)]
            try:
                response = client.chat.completions.create(
                    model="llama-3.3-70b", messages=messages, max_tokens=500, temperature=0.0,
                )
                raw = response.choices[0].message.content
                parsed = extract_json(raw)
                if is_structurally_valid(parsed):
                    valid = True
                    break
                print(f"[{i} attempt {attempt+1}: JSON غیرِساختاری، ارسالِ مجدد...")
            except Exception as e:
                print(f"[{i} attempt {attempt+1}: خطا: {e}")
                time.sleep(2 * (attempt + 1))

        record = {"sample_key": key, "poem_id": row["poem_id"], "poet": row["poet"],
                   "split": row["split"], "k": row["k"], "raw_response": raw, "valid": valid}
        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        out_f.flush()

        if not valid:
            print(f"[{i} شکست خورد بعد از {MAX_RETRIES} تلاش")
        if (i + 1) % 50 == 0:
            print(f"Progress: {i+1}/{len(llm_sample)}")

print("Done:", raw_output_path)

In [ ]:
from collections import defaultdict

results, still_invalid, hallucination_stats = [], 0, []

with open(raw_output_path, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        candidates = closed_set_poets if rec["split"] == "test_closed" else open_set_poets
        cand_norm = set(normalize_name(c) for c in candidates)
        parsed = extract_json(rec["raw_response"])
        if parsed is None or not is_structurally_valid(parsed):
            still_invalid += 1
            rank = None
        else:
            ranked_norm = [normalize_name(a) for a in parsed["ranked_authors"]]
            true_norm = normalize_name(rec["poet"])
            rank = ranked_norm.index(true_norm) if true_norm in ranked_norm else None
            hallucination_stats.append(sum(1 for a in ranked_norm if a in cand_norm) / len(ranked_norm))
        results.append({**rec, "rank": rank})

print(f"Total: {len(results)} | Structurally invalid: {still_invalid}")
avg_compliance = (sum(hallucination_stats) / len(hallucination_stats)) if hallucination_stats else None
if avg_compliance is not None:
    print(f"میانگینِ پایبندی به فهرستِ نامزدها: {avg_compliance*100:.1f}%")

groups = defaultdict(list)
for r in results:
    groups[(r["split"], r["k"])].append(r["rank"])

def compute_metrics(ranks):
    n = len(ranks)
    recall_at = {1: 0, 5: 0, 10: 0}
    mrr_sum = 0.0
    for rank in ranks:
        if rank is not None:
            for k_cut in (1, 5, 10):
                if rank < k_cut:
                    recall_at[k_cut] += 1
            mrr_sum += 1.0 / (rank + 1)
    return {"n": n, "recall@1": recall_at[1]/n, "recall@5": recall_at[5]/n, "recall@10": recall_at[10]/n, "mrr": mrr_sum/n}

print("\n=== Llama-3.3-70B Zero-Shot — Recall@1/5/10 + MRR ===")
for (split, k) in sorted(groups.keys()):
    m = compute_metrics(groups[(split, k)])
    print(f"{split} | k={k} | n={m['n']} | R@1={m['recall@1']*100:.2f}% | R@5={m['recall@5']*100:.2f}% | R@10={m['recall@10']*100:.2f}% | MRR={m['mrr']:.3f}")

metrics_path = os.path.join(LLM_DIR, "llama_zeroshot_metrics.json")
metrics_summary = {f"{split}_k{k}": compute_metrics(groups[(split, k)]) for (split, k) in groups}
metrics_summary["_meta"] = {"avg_candidate_compliance": avg_compliance, "structurally_invalid": still_invalid}
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)
print(f"\nSaved metrics to {metrics_path}")

In [ ]:
import time
from openai import OpenAI, RateLimitError, AuthenticationError, APIStatusError, APIConnectionError

API_KEYS = [
]

BASE_URL = "https://api.tokenmix.ai/v1"
MODEL_NAME = "deepseek-v4-pro"

key_clients = {k: OpenAI(base_url=BASE_URL, api_key=k) for k in API_KEYS}
active_keys = list(API_KEYS)
_key_cursor = {"i": 0}

EXHAUSTION_SIGNALS = [
    "insufficient_quota", "exceeded your current quota", "billing",
    "invalid_api_key", "incorrect api key", "account is not active",
    "deactivated", "account_deactivated",
]

def _is_exhaustion_error(e) -> bool:
    msg = str(e).lower()
    return any(sig in msg for sig in EXHAUSTION_SIGNALS)

def _next_key() -> str:
    if not active_keys:
        raise RuntimeError("همه‌ی کلیدهای API غیرفعال/ته‌کشیده شدند. لطفاً کلید جدید اضافه کنید.")
    key = active_keys[_key_cursor["i"] % len(active_keys)]
    _key_cursor["i"] += 1
    return key

def call_llm_with_key_rotation(messages, max_tokens=1500, max_key_attempts=None):
    max_key_attempts = max_key_attempts or max(len(active_keys), 1)
    attempts, last_error = 0, None
    while attempts < max_key_attempts and active_keys:
        key = _next_key()
        client = key_clients[key]
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                max_tokens=max_tokens,
                extra_body={"reasoning_effort": "none"},
            )
            return response.choices[0].message.content
        except RateLimitError as e:
            if _is_exhaustion_error(e):
                print(f"⚠️ کلید ...{key[-6:]} ته‌کشیده/غیرفعال شد — برای همیشه از چرخه حذف شد.")
                if key in active_keys:
                    active_keys.remove(key)
            else:
                time.sleep(2)
            last_error = e
        except AuthenticationError as e:
            print(f"⚠️ کلید ...{key[-6:]} نامعتبر است — از چرخه حذف شد.")
            if key in active_keys:
                active_keys.remove(key)
            last_error = e
        except (APIConnectionError, APIStatusError) as e:
            time.sleep(1)
            last_error = e
        attempts += 1
    raise RuntimeError(f"همه‌ی تلاش‌ها با کلیدهای فعال شکست خورد. آخرین خطا: {last_error}")

In [ ]:
deepseek_MAX_RETRIES = 3
deepseek_raw_output_path = os.path.join(LLM_DIR, "deepseek_v4pro_zeroshot_raw.jsonl")

already_done_keys = set()
if os.path.exists(deepseek_raw_output_path):
    with open(deepseek_raw_output_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                already_done_keys.add(rec["sample_key"])
            except json.JSONDecodeError:
                continue

remaining_rows = [row for row in llm_sample if make_sample_key(row) not in already_done_keys]
print(f"قبلاً پردازش‌شده: {len(already_done_keys)} | باقی‌مانده: {len(remaining_rows)} از {len(llm_sample)}")

structurally_invalid_count = 0
candidate_compliance_scores = []

with open(deepseek_raw_output_path, "a", encoding="utf-8") as f:
    for idx, row in enumerate(remaining_rows):
        messages = build_messages(row)
        candidates = build_candidate_list(row)

        parsed = None
        for attempt in range(deepseek_MAX_RETRIES):
            try:
                raw_text = call_llm_with_key_rotation(messages)
                parsed = extract_json(raw_text)
                if is_structurally_valid(parsed):
                    break
                parsed = None
            except Exception as e:
                print(f"[{idx} خطا در تلاشِ {attempt+1}: {e}", flush=True)
                parsed = None

        if parsed is None:
            structurally_invalid_count += 1
            record = {"sample_key": make_sample_key(row), "ranked_authors": [], "valid": False}
        else:
            ranked = [normalize_name(n) for n in parsed["ranked_authors"]]
            compliance = sum(1 for n in ranked if n in candidates) / max(len(ranked), 1)
            candidate_compliance_scores.append(compliance)
            record = {
                "sample_key": make_sample_key(row),
                "ranked_authors": ranked,
                "true_poet": row["poet"],
                "valid": True,
            }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()

        if idx % 20 == 0:
            print(f"{idx}/{len(remaining_rows)} (باقی‌مانده) — کلیدهایِ فعال: {len(active_keys)}", flush=True)

print(f"این اجرا: {idx+1} نمونه‌ی جدید پردازش شد | ساختاری‌نامعتبر: {structurally_invalid_count}")
if candidate_compliance_scores:
    print(f"میانگینِ پایبندی به فهرستِ نامزدها (این اجرا): {sum(candidate_compliance_scores)/len(candidate_compliance_scores)*100:.1f}%")

In [ ]:
from collections import defaultdict

sample_key_to_row = {make_sample_key(row): row for row in llm_sample}

results, still_invalid, hallucination_stats = [], 0, []

with open(deepseek_raw_output_path, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        row = sample_key_to_row.get(rec["sample_key"])
        if row is None:
            continue
        split, k = row["split"], row["k"]
        candidates = closed_set_poets if split == "test_closed" else open_set_poets
        cand_norm = set(normalize_name(c) for c in candidates)

        if not rec.get("valid") or not rec.get("ranked_authors"):
            still_invalid += 1
            rank = None
        else:
            ranked_norm = [normalize_name(a) for a in rec["ranked_authors"]]
            true_norm = normalize_name(rec["true_poet"])
            rank = ranked_norm.index(true_norm) if true_norm in ranked_norm else None
            hallucination_stats.append(sum(1 for a in ranked_norm if a in cand_norm) / len(ranked_norm))

        results.append({**rec, "split": split, "k": k, "rank": rank})

print(f"Total: {len(results)} | Structurally invalid: {still_invalid}")
avg_compliance = (sum(hallucination_stats) / len(hallucination_stats)) if hallucination_stats else None
if avg_compliance is not None:
    print(f"میانگینِ پایبندی به فهرستِ نامزدها: {avg_compliance*100:.1f}%")

groups = defaultdict(list)
for r in results:
    groups[(r["split"], r["k"])].append(r["rank"])

def compute_metrics(ranks):
    n = len(ranks)
    recall_at = {1: 0, 5: 0, 10: 0}
    mrr_sum = 0.0
    for rank in ranks:
        if rank is not None:
            for k_cut in (1, 5, 10):
                if rank < k_cut:
                    recall_at[k_cut] += 1
            mrr_sum += 1.0 / (rank + 1)
    return {"n": n, "recall@1": recall_at[1]/n, "recall@5": recall_at[5]/n, "recall@10": recall_at[10]/n, "mrr": mrr_sum/n}

print("\n=== DeepSeek-V4-Pro Zero-Shot — Recall@1/5/10 + MRR ===")
for (split, k) in sorted(groups.keys()):
    m = compute_metrics(groups[(split, k)])
    print(f"{split} | k={k} | n={m['n']} | R@1={m['recall@1']*100:.2f}% | R@5={m['recall@5']*100:.2f}% | R@10={m['recall@10']*100:.2f}% | MRR={m['mrr']:.3f}")

deepseek_metrics_path = os.path.join(LLM_DIR, "deepseek_v4pro_zeroshot_metrics.json")
metrics_summary = {f"{split}_k{k}": compute_metrics(groups[(split, k)]) for (split, k) in groups}
metrics_summary["_meta"] = {"avg_candidate_compliance": avg_compliance, "structurally_invalid": still_invalid}
with open(deepseek_metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

print(f"\nSaved metrics to {deepseek_metrics_path}")

In [ ]:
matched = [r for r in results if r["rank"] is not None][:8]
for r in matched:
    print(f"true_poet: {r['true_poet']!r} | normalize_name: {normalize_name(r['true_poet'])!r}")
    print(f"rank: {r['rank']} | ranked_authors[:3]: {r['ranked_authors'][:3]}")
    print("-" * 60)

In [ ]:
import random

random.seed(42)

train_path = os.path.join(PROJECT_DIR, "phase1_outputs", "train.jsonl")
with open(train_path, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

print("تعداد شعر در train:", len(train_data))

def extract_k_bait_excerpt(poem, k):
    lines = poem["text"].split("\n")
    num_baits = poem.get("num_baits") or 0
    if num_baits == 0:
        return None
    lines_per_bait = max(1, len(lines) // num_baits)
    needed_lines = k * lines_per_bait
    if needed_lines > len(lines):
        return None
    return "\n".join(lines[:needed_lines])

def build_fewshot_set(k, n_shots=5, seed=42):
    pool = [p for p in train_data if (p.get("num_baits") or 0) >= k]
    rng = random.Random(seed + k)
    rng.shuffle(pool)
    chosen, used_poets = [], set()
    for poem in pool:
        poet = poem["poet"]
        if poet in used_poets or poet not in closed_set_poets:
            continue
        excerpt = extract_k_bait_excerpt(poem, k)
        if not excerpt:
            continue
        chosen.append({"poet": poet, "excerpt": excerpt})
        used_poets.add(poet)
        if len(chosen) == n_shots:
            break
    return chosen

FEWSHOT_SETS = {k: build_fewshot_set(k) for k in (1, 2, 4, 6)}
for k, shots in FEWSHOT_SETS.items():
    print(f"k={k}: " + ", ".join(s["poet"] for s in shots))

def build_demo_answer(true_poet, seed_offset):
    rng = random.Random(1000 + seed_offset)
    fillers = rng.sample([p for p in closed_set_poets if p != true_poet], 9)
    ranked = [true_poet] + fillers
    demo_json = {
        "ranked_authors": ranked,
        "rationale": "وزن، قافیه و واژگانِ به‌کاررفته با سبکِ شناخته‌شده‌ی این شاعر هم‌خوانی دارد."
    }
    return json.dumps(demo_json, ensure_ascii=False)

def build_messages_fewshot(row):
    candidates = build_candidate_list(row)
    candidate_str = format_candidates(candidates)
    k = row["k"]

    system_msg = (
        "You are an expert in Persian (Farsi) classical poetry stylometry. "
        "Given a short excerpt from a Persian poem and a numbered list of candidate poets, "
        "identify which poet most likely wrote it, based on subtle stylistic cues: meter (وزن), "
        "rhyme/refrain (قافیه/ردیف), diction and vocabulary, era-specific language, and thematic tone.\n\n"
        "Respond with STRICTLY VALID JSON ONLY. No markdown, no code fences, no text before or after "
        "the JSON object. Use exactly this schema:\n"
        '{\n  "ranked_authors": ["<name>", "... exactly 10 names total"],\n  "rationale": "1-3 sentences"\n}\n\n'
        "Rules:\n"
        "- \"ranked_authors\" must contain EXACTLY 10 distinct names, ordered from most to least likely.\n"
        "- Every name MUST be copied character-for-character from the numbered candidate list below.\n"
        "- Do not invent names that are not in the candidate list.\n"
        "- Do not repeat any name."
    )

    messages = [{"role": "system", "content": system_msg}]

    demo_candidate_str = format_candidates(closed_set_poets)
    messages.append({
        "role": "user",
        "content": (
            f"[Training examples below all share this candidate list ({len(closed_set_poets)} poets). "
            f"This list is ONLY for the training examples, not for your final answer.]\n{demo_candidate_str}"
        ),
    })
    messages.append({"role": "assistant", "content": "Understood."})

    for i, shot in enumerate(FEWSHOT_SETS[k]):
        demo_user_msg = (
            f"Training example {i+1} — Persian poem excerpt ({k} couplet(s)):\n\"\"\"\n{shot['excerpt']}\n\"\"\"\n\n"
            "Task: Rank the 10 most likely authors from the candidate list above (most likely first), "
            "and briefly explain the stylistic rationale for your top choice."
        )
        messages.append({"role": "user", "content": demo_user_msg})
        messages.append({"role": "assistant", "content": build_demo_answer(shot["poet"], seed_offset=i * 100 + k)})

    user_msg = (
        f"[Now for your actual task, use this NEW candidate list ({len(candidates)} poets) — "
        f"do NOT reuse names from the training examples' list above unless they also appear here.]\n{candidate_str}\n\n"
        f"Persian poem excerpt ({row['k']} couplet(s)):\n\"\"\"\n{row['text']}\n\"\"\"\n\n"
        "Task: Rank the 10 most likely authors from the candidate list above (most likely first), "
        "and briefly explain the stylistic rationale for your top choice."
    )
    messages.append({"role": "user", "content": user_msg})

    return messages

In [ ]:
deepseek_5shot_raw_output_path = os.path.join(LLM_DIR, "deepseek_v4pro_5shot_raw.jsonl")

already_done_keys_5shot = set()
if os.path.exists(deepseek_5shot_raw_output_path):
    with open(deepseek_5shot_raw_output_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                already_done_keys_5shot.add(rec["sample_key"])
            except json.JSONDecodeError:
                continue

remaining_rows_5shot = [row for row in llm_sample if make_sample_key(row) not in already_done_keys_5shot]
print(f"قبلاً پردازش‌شده: {len(already_done_keys_5shot)} | باقی‌مانده: {len(remaining_rows_5shot)} از {len(llm_sample)}")

structurally_invalid_count_5shot = 0
candidate_compliance_scores_5shot = []

with open(deepseek_5shot_raw_output_path, "a", encoding="utf-8") as f:
    for idx, row in enumerate(remaining_rows_5shot):
        messages = build_messages_fewshot(row)
        candidates = build_candidate_list(row)
        parsed = None
        for attempt in range(deepseek_MAX_RETRIES):
            try:
                raw_text = call_llm_with_key_rotation(messages)
                parsed = extract_json(raw_text)
                if is_structurally_valid(parsed):
                    break
                parsed = None
            except Exception as e:
                print(f"[{idx} خطا در تلاشِ {attempt+1}: {e}", flush=True)
                parsed = None
        if parsed is None:
            structurally_invalid_count_5shot += 1
            record = {"sample_key": make_sample_key(row), "ranked_authors": [], "valid": False}
        else:
            ranked = [normalize_name(n) for n in parsed["ranked_authors"]]
            compliance = sum(1 for n in ranked if n in candidates) / max(len(ranked), 1)
            candidate_compliance_scores_5shot.append(compliance)
            record = {
                "sample_key": make_sample_key(row),
                "ranked_authors": ranked,
                "true_poet": row["poet"],
                "valid": True,
            }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        if idx % 20 == 0:
            print(f"{idx}/{len(remaining_rows_5shot)} (باقی‌مانده) — کلیدهایِ فعال: {len(active_keys)}", flush=True)

print(f"این اجرا: {idx+1} نمونه‌ی جدید پردازش شد | ساختاری‌نامعتبر: {structurally_invalid_count_5shot}")
if candidate_compliance_scores_5shot:
    print(f"میانگینِ پایبندی به فهرستِ نامزدها (این اجرا): {sum(candidate_compliance_scores_5shot)/len(candidate_compliance_scores_5shot)*100:.1f}%")

In [ ]:
from collections import defaultdict

sample_key_to_row = {make_sample_key(row): row for row in llm_sample}
results_5shot, still_invalid_5shot, hallucination_stats_5shot = [], 0, []

with open(deepseek_5shot_raw_output_path, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        row = sample_key_to_row.get(rec["sample_key"])
        if row is None:
            continue
        split, k = row["split"], row["k"]
        candidates = closed_set_poets if split == "test_closed" else open_set_poets
        cand_norm = set(normalize_name(c) for c in candidates)
        if not rec.get("valid") or not rec.get("ranked_authors"):
            still_invalid_5shot += 1
            rank = None
        else:
            ranked_norm = [normalize_name(a) for a in rec["ranked_authors"]]
            true_norm = normalize_name(rec["true_poet"])
            rank = ranked_norm.index(true_norm) if true_norm in ranked_norm else None
            hallucination_stats_5shot.append(sum(1 for a in ranked_norm if a in cand_norm) / len(ranked_norm))
        results_5shot.append({**rec, "split": split, "k": k, "rank": rank})

print(f"Total: {len(results_5shot)} | Structurally invalid: {still_invalid_5shot}")
avg_compliance_5shot = (sum(hallucination_stats_5shot) / len(hallucination_stats_5shot)) if hallucination_stats_5shot else None
if avg_compliance_5shot is not None:
    print(f"میانگینِ پایبندی به فهرستِ نامزدها: {avg_compliance_5shot*100:.1f}%")

groups_5shot = defaultdict(list)
for r in results_5shot:
    groups_5shot[(r["split"], r["k"])].append(r["rank"])

print("\n=== DeepSeek-V4-Pro 5-Shot — Recall@1/5/10 + MRR ===")
for (split, k) in sorted(groups_5shot.keys()):
    m = compute_metrics(groups_5shot[(split, k)])
    print(f"{split} | k={k} | n={m['n']} | R@1={m['recall@1']*100:.2f}% | R@5={m['recall@5']*100:.2f}% | R@10={m['recall@10']*100:.2f}% | MRR={m['mrr']:.3f}")

deepseek_5shot_metrics_path = os.path.join(LLM_DIR, "deepseek_v4pro_5shot_metrics.json")
metrics_summary_5shot = {f"{split}_k{k}": compute_metrics(groups_5shot[(split, k)]) for (split, k) in groups_5shot}
metrics_summary_5shot["_meta"] = {"avg_candidate_compliance": avg_compliance_5shot, "structurally_invalid": still_invalid_5shot}
with open(deepseek_5shot_metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary_5shot, f, ensure_ascii=False, indent=2)

print(f"\nSaved metrics to {deepseek_5shot_metrics_path}")